<a href="https://colab.research.google.com/github/pajola/secai-course/blob/main/module-01-adversarial-ml/02_adversarial_text.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" height="32"></a>

# 2 · Adversarial Examples on Text — typos, triggers, transfer, defenses

**SecAI course · Adversarial Machine Learning, part 2**

Images live in a continuous space: "move every pixel by ε" is a natural perturbation. Text does not:

* the input is **discrete** (tokens) — there is no gradient *into* the input, only into embeddings;
* the perturbation must preserve **meaning** (for the human) and **plausibility** (not look obviously broken);
* so attacks work at the **character** level (typos, homoglyphs), the **word** level (synonyms, deletions), or the **sentence** level (paraphrase, appended "triggers").

We attack a sentiment classifier on SST-2 (short movie-review sentences). The victim is deliberately simple — TF-IDF + logistic regression — so that every step is inspectable; the same ideas (TextBugger, TextFooler, HotFlip, universal triggers) apply to transformers.

| | Evasion attack on text |
|---|---|
| **Goal** | flip the predicted sentiment |
| **Knowledge** | white-box (weights) → then score-based black-box (only the output probabilities) |
| **Capability** | edit at most *k* words; each edit must keep the sentence readable |

In [1]:
# Setup — on Google Colab, fetch the course helper module and install PhantomText (all other packages are preinstalled).
import os, sys, subprocess, urllib.request
if "google.colab" in sys.modules:
    if not os.path.exists("secai_utils.py"):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/pajola/secai-course/main/module-01-adversarial-ml/secai_utils.py", "secai_utils.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "phantomtext"], check=True)

In [2]:
import numpy as np, matplotlib.pyplot as plt, re, random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from secai_utils import load_sst2

random.seed(0); np.random.seed(0)
train_texts, y_train, test_texts, y_test = load_sst2()
print(len(train_texts), "train /", len(test_texts), "test sentences")
for t, l in list(zip(test_texts, y_test))[:4]:
    print(f"[{'POS' if l else 'NEG'}] {t}")

6920 train / 1821 test sentences
[NEG] no movement , no yuks , not much of anything
[NEG] a gob of drivel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr me br l e
[NEG] gangs of new york is an unapologetic mess , whose only saving grace is that it ends by blowing just about everything up
[NEG] we never really feel involved with the story , as all of its ideas remain just that abstract ideas


## 2.1 The victim

In [3]:
vec = TfidfVectorizer(lowercase=True, token_pattern=r"[a-z']+")
clf = LogisticRegression(C=3.0, max_iter=2000)
victim = make_pipeline(vec, clf).fit(train_texts, y_train)

def acc(model, texts, labels): return (model.predict(texts) == labels).mean()
print(f"clean test accuracy: {acc(victim, test_texts, y_test):.3f}")

vocab = np.array(vec.get_feature_names_out())
coef = clf.coef_[0]
print("most positive words:", list(vocab[np.argsort(coef)[-10:]][::-1]))
print("most negative words:", list(vocab[np.argsort(coef)[:10]]))

clean test accuracy: 0.815
most positive words: ['fun', 'best', 'solid', 'powerful', 'entertaining', 'and', 'works', 'enjoyable', 'love', 'heart']
most negative words: ['too', 'bad', "n't", 'worst', 'dull', 'feels', 'suffers', 'stupid', 'mess', 'clich']


## 2.2 Character-level attack (TextBugger-style)

Idea: **(1)** find the words that matter most for the current prediction, **(2)** corrupt them just enough that the tokenizer no longer recognises them (so they fall out of the vocabulary and contribute nothing), **(3)** stop as soon as the label flips.

Three "bugs" that humans read through without noticing: an adjacent-character **swap** (`terrible → terrilbe`), an **inserted** character (`terri-ble`), and a **homoglyph** — replace a Latin letter with a visually identical Unicode one (`a` → Cyrillic `а`). The last one is invisible to the eye and devastating for any system that matches strings.

In [4]:
HOMOGLYPHS = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х", "i": "і"}

def bug_swap(w):
    if len(w) < 4: return None
    i = random.randrange(1, len(w) - 2); return w[:i] + w[i+1] + w[i] + w[i+2:]
def bug_insert(w):
    if len(w) < 3: return None
    i = random.randrange(1, len(w) - 1); return w[:i] + "-" + w[i:]
def bug_homoglyph(w):
    for ch in w:
        if ch in HOMOGLYPHS: return w.replace(ch, HOMOGLYPHS[ch], 1)
    return None
BUGS = [bug_swap, bug_insert, bug_homoglyph]

def tokens(text): return text.split()

def importance_whitebox(text, label):
    # contribution of each word to the *true* class  =  coef[word] * tfidf(word)   (sign-adjusted for class 0)
    x = vec.transform([text]); sign = 1 if label == 1 else -1
    contrib = {vocab[j]: sign * coef[j] * x[0, j] for j in x.indices}
    return [contrib.get(w.lower().strip("'"), 0.0) for w in tokens(text)]

def importance_blackbox(model, text, label):
    # leave-one-out: how much does P(true class) drop if we delete the word?  (only needs model.predict_proba)
    toks = tokens(text); p0 = model.predict_proba([text])[0, label]
    return [p0 - model.predict_proba([" ".join(toks[:i] + toks[i+1:])])[0, label] for i in range(len(toks))]

def attack_text(model, text, label, importance, max_edits=3):
    toks = tokens(text)
    order = np.argsort(importance(text, label))[::-1]       # most important words first
    edits = 0
    for i in order:
        if edits >= max_edits or model.predict([" ".join(toks)])[0] != label:
            break
        candidates = [b(toks[i]) for b in BUGS]
        candidates = [c for c in candidates if c]
        if not candidates: continue
        # score-based choice: pick the bug that lowers P(true class) the most
        scores = [model.predict_proba([" ".join(toks[:i] + [c] + toks[i+1:])])[0, label] for c in candidates]
        toks[i] = candidates[int(np.argmin(scores))]; edits += 1
    adv = " ".join(toks)
    return adv, model.predict([adv])[0] != label, edits

# demo on a few correctly classified sentences
idx = [i for i in range(len(test_texts)) if victim.predict([test_texts[i]])[0] == y_test[i]]
for i in idx[:6]:
    adv, ok, n = attack_text(victim, test_texts[i], y_test[i], lambda t, l: importance_whitebox(t, l))
    print(f"[{'POS' if y_test[i] else 'NEG'}] {test_texts[i]}\n   -> {adv}\n   flipped={ok} with {n} edit(s)\n")

[NEG] no movement , no yuks , not much of anything
   -> nо movement , nо yuks , not much of anytihng
   flipped=False with 3 edit(s)

[NEG] a gob of drivel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr me br l e
   -> a gob of dirvel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr mе br l e
   flipped=True with 2 edit(s)

[NEG] gangs of new york is an unapologetic mess , whose only saving grace is that it ends by blowing just about everything up
   -> gangs of new york is an unapologetic mses , whose olny saving grace is that it ends by blowing just about everything up
   flipped=True with 2 edit(s)

[NEG] we never really feel involved with the story , as all of its ideas remain just that abstract ideas
   -> we never re-ally feel involved with the story , as all of its ideas remain ju-st that abstract ideas
   flipped=True with 2 edit(s)

[POS] this is one of polans

Look closely at the homoglyph edits: the adversarial sentence is **byte-different but pixel-identical** to the original. (This is why input *sanitisation* — Unicode normalisation, confusable detection — is a security control, not a text-cleaning nicety.)

Now the success rate over the test set, with white-box vs. black-box word importance.

In [5]:
def attack_success_rate(model, importance, n=300, max_edits=3, seed=0):
    random.seed(seed)
    sample = idx[:n]; flips, total_edits = 0, 0
    advs = []
    for i in sample:
        adv, ok, k = attack_text(model, test_texts[i], y_test[i], importance, max_edits)
        flips += ok; total_edits += k; advs.append(adv)
    return flips / len(sample), total_edits / len(sample), advs

asr_wb, edits_wb, advs_wb = attack_success_rate(victim, importance_whitebox)
asr_bb, edits_bb, advs_bb = attack_success_rate(victim, lambda t, l: importance_blackbox(victim, t, l))
print(f"white-box importance:  success rate = {asr_wb:.2f}   avg edits = {edits_wb:.2f}  (max 3)")
print(f"black-box importance:  success rate = {asr_bb:.2f}   avg edits = {edits_bb:.2f}  (max 3)")

white-box importance:  success rate = 0.68   avg edits = 2.29  (max 3)
black-box importance:  success rate = 0.67   avg edits = 2.27  (max 3)


Three typos flip the majority of predictions — and the black-box attacker, who only sees output probabilities, does about as well as the one with the weights (the "which word matters" question is cheap to answer by querying).

## 2.3 Universal adversarial triggers (Wallace et al., 2019)

Instead of crafting a perturbation per input, find **one short string** that, prepended to *any* negative review, makes the model say "positive". For a linear model the recipe is embarrassingly simple: take the words with the largest positive weights. For neural models the same thing is done with gradient-guided search over the embedding space.

In [6]:
neg_idx = [i for i in idx if y_test[i] == 0]
neg_texts = [test_texts[i] for i in neg_idx]

for k in [1, 2, 3, 5]:
    trigger = " ".join(vocab[np.argsort(coef)[-k:]][::-1])
    flipped = (victim.predict([trigger + " " + t for t in neg_texts]) == 1).mean()
    print(f"trigger = {trigger!r:40s}  ->  {flipped:.0%} of negative reviews now 'positive'")

print("\nexample:", neg_texts[0], "\n   ->", trigger + " " + neg_texts[0])

trigger = 'fun'                                     ->  49% of negative reviews now 'positive'
trigger = 'fun best'                                ->  81% of negative reviews now 'positive'
trigger = 'fun best solid'                          ->  96% of negative reviews now 'positive'
trigger = 'fun best solid powerful entertaining'    ->  100% of negative reviews now 'positive'

example: no movement , no yuks , not much of anything 
   -> fun best solid powerful entertaining no movement , no yuks , not much of anything


Three words are enough to flip almost every negative review. The modern descendant of this attack is **prompt injection** against LLMs: an input-independent string that hijacks the model's behaviour regardless of the surrounding content.

## 2.4 Transferability

Same experiment as for images: craft the typo attacks on the LR victim, then feed them to models the attacker never saw.

In [7]:
models = {
    "LR word TF-IDF (victim)":       victim,
    "Naive Bayes, word counts":      make_pipeline(TfidfVectorizer(token_pattern=r"[a-z']+", use_idf=False, norm=None), MultinomialNB()).fit(train_texts, y_train),
    "LR, word+bigram TF-IDF":        make_pipeline(TfidfVectorizer(token_pattern=r"[a-z']+", ngram_range=(1, 2)), LogisticRegression(C=3, max_iter=2000)).fit(train_texts, y_train),
    "LR, char 2-5-gram TF-IDF":      make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), LogisticRegression(C=3, max_iter=2000)).fit(train_texts, y_train),
}
sample_labels = y_test[idx[:300]]
clean_sample  = [test_texts[i] for i in idx[:300]]
print("(evaluated on 300 test sentences the victim classifies correctly, hence its clean accuracy of 1.0)")
print(f"{'model':32s} {'clean acc':>10s} {'acc on transferred adv':>24s}")
for name, m in models.items():
    print(f"{name:32s} {acc(m, clean_sample, sample_labels):10.3f} {acc(m, advs_wb, sample_labels):24.3f}")

(evaluated on 300 test sentences the victim classifies correctly, hence its clean accuracy of 1.0)
model                             clean acc   acc on transferred adv
LR word TF-IDF (victim)               1.000                    0.320
Naive Bayes, word counts              0.940                    0.563
LR, word+bigram TF-IDF                0.963                    0.577


LR, char 2-5-gram TF-IDF              0.943                    0.730


The typos were chosen against the word-level LR, yet they also hurt Naive Bayes and the bigram model: anything that tokenises on whitespace loses the same words. The **character n-gram** model is the odd one out — a hint for the defender.

## 2.5 Defenses

1. **Robust features** — character n-grams see `terri-ble` and `terrible` as mostly the same thing (already visible above).
2. **Input sanitisation** with [**PhantomText**](https://github.com/lucapajola/PhantomText) (`pip install phantomtext`) — a toolkit that **detects** and **removes** the four Unicode tricks used to hide text from machines while keeping it readable for humans: homoglyphs, zero-width characters, diacritical marks and bidi overrides. We add spell-correction on top, mapping out-of-vocabulary words back to the nearest vocabulary word, to undo the *visible* typos (swaps, hyphens) that no Unicode check can see.
3. **Adversarial data augmentation** — train on bugged copies of the training sentences so the model learns the bugged forms too.

Each is evaluated with the attacker **re-running** against the defended model (adaptive), not just replaying old adversarial examples.

### Detection first: does PhantomText notice the attack?

Before cleaning anything, a defender can simply **flag** suspicious inputs. Each PhantomText technique class has a `check(text)` detector and a `sanitized(text)` cleaner; we run all four on the adversarial sentences from §2.2 and on the clean test set.

In [8]:
from phantomtext.obfuscation.homoglyph_text import HomoglyphText          # Unicode confusables (downloaded from unicode.org on first use)
from phantomtext.obfuscation.zero_width_text import ZeroWidthText         # ZWSP, ZWNJ, ZWJ, ...
from phantomtext.obfuscation.diacritical_marks import DiacriticalMarks    # combining marks
from phantomtext.obfuscation.reordering_char import BidiText              # bidi overrides (LRO, RLO, ...)

PHANTOM = [HomoglyphText(), ZeroWidthText(), DiacriticalMarks(), BidiText()]

def detect(text):    # names of the Unicode tricks found in `text`
    return [type(t).__name__ for t in PHANTOM if t.check(text)]

def sanitize(text):  # undo all of them
    for t in PHANTOM: text = t.sanitized(text)
    return text

has_homoglyph = [any(c in HOMOGLYPHS.values() for c in a) for a in advs_wb]
flagged       = [bool(detect(a)) for a in advs_wb]
print(f"adversarial sentences that use a homoglyph edit:   {np.mean(has_homoglyph):.0%}")
print(f"adversarial sentences flagged by PhantomText:      {np.mean(flagged):.0%}")
print(f"clean test sentences flagged (false positives):    {np.mean([bool(detect(t)) for t in test_texts]):.1%}")

missed = [a for a, h, f in zip(advs_wb, has_homoglyph, flagged) if h and not f]
caught = next(a for a, f in zip(advs_wb, flagged) if f)
print("\ncaught:   ", caught, " ", detect(caught), "\nsanitized:", sanitize(caught))
if missed:
    odd = sorted({c for a in missed for c in a if ord(c) > 127})
    print(f"\nmissed {len(missed)} homoglyph sentence(s); characters not in PhantomText's table:", [f"{c} U+{ord(c):04X}" for c in odd])

adversarial sentences that use a homoglyph edit:   61%
adversarial sentences flagged by PhantomText:      44%
clean test sentences flagged (false positives):    0.0%

caught:    a gob of dirvel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr mе br l e   ['HomoglyphText'] 
sanitized: a gob of dirvel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr me br l e

missed 52 homoglyph sentence(s); characters not in PhantomText's table: ['о U+043E']


Two things to notice:

* **Detection is cheap and precise for the invisible tricks** — no false positives on clean text — but it is **blind to visible typos** (`terrilbe`, `terri-ble`): those are not Unicode anomalies, so a typo-only attack goes straight through. Detection and sanitisation complement, not replace, robust models.
* **A sanitiser is only as good as its confusables table.** PhantomText uses Unicode's curated *intentional* confusables list, which maps `o` only to Greek omicron `ο` — the Cyrillic `о` our attack uses slips through. The full `confusables.txt` covers it at the price of more false positives. Whatever list you pick, an adaptive attacker will look for the characters it misses.

Now the full comparison, with the attacker adapting to each defense:

In [9]:
import difflib

train_vocab = set(vec.get_feature_names_out())
_by_key = {}                                            # index the vocabulary by (first letter, length) to keep spell-correction fast
for w in train_vocab:
    for dl in (-1, 0, 1): _by_key.setdefault((w[0], len(w) + dl), []).append(w)
_cache = {}
def spell_correct(lw):
    if lw not in _cache:
        close = difflib.get_close_matches(lw, _by_key.get((lw[0], len(lw)), []), n=1, cutoff=0.8)
        _cache[lw] = close[0] if close else lw
    return _cache[lw]

def normalise(text, spell=True):
    text = sanitize(text)                                # PhantomText: undo homoglyphs, zero-width, diacritics, bidi
    if not spell: return text
    out = []
    for w in text.replace("-", "").split():
        lw = w.lower()
        if lw in train_vocab or not lw.isalpha(): out.append(w); continue
        out.append(spell_correct(lw))
    return " ".join(out)

class Normalised:
    # wrap any sklearn pipeline with an input-sanitisation step
    def __init__(self, model, spell=True): self.model, self.spell = model, spell
    def predict(self, texts): return self.model.predict([normalise(t, self.spell) for t in texts])
    def predict_proba(self, texts): return self.model.predict_proba([normalise(t, self.spell) for t in texts])

def augment(texts, labels, n_bugs=2, seed=0):
    random.seed(seed); out_t, out_l = list(texts), list(labels)
    for t, l in zip(texts, labels):
        toks = t.split()
        for i in random.sample(range(len(toks)), min(n_bugs, len(toks))):
            b = random.choice(BUGS)(toks[i]); toks[i] = b or toks[i]
        out_t.append(" ".join(toks)); out_l.append(l)
    return out_t, np.array(out_l)

aug_texts, aug_labels = augment(train_texts, y_train)
defended = {
    "no defense (victim)":           victim,
    "char n-gram features":          models["LR, char 2-5-gram TF-IDF"],
    "PhantomText sanitisation":      Normalised(victim, spell=False),
    "PhantomText + spell-correction": Normalised(victim),
    "adversarial augmentation":      make_pipeline(TfidfVectorizer(token_pattern=r"[a-z']+"), LogisticRegression(C=3, max_iter=2000)).fit(aug_texts, aug_labels),
    "PhantomText + spell + char n-grams": Normalised(models["LR, char 2-5-gram TF-IDF"]),
}
print(f"{'defense':36s} {'clean acc':>10s} {'adaptive attack success':>26s}")
for name, m in defended.items():
    asr, _, _ = attack_success_rate(m, lambda t, l: importance_blackbox(m, t, l), n=200)
    print(f"{name:36s} {acc(m, test_texts, y_test):10.3f} {asr:26.2f}")

defense                               clean acc    adaptive attack success


no defense (victim)                       0.815                       0.67


char n-gram features                      0.823                       0.44


PhantomText sanitisation                  0.815                       0.64


PhantomText + spell-correction            0.815                       0.21


adversarial augmentation                  0.813                       0.66


PhantomText + spell + char n-grams        0.819                       0.19


Three lessons hide in this table:

* **Random augmentation ≠ adversarial training.** Training on randomly bugged copies barely moves the needle, because the attacker does not pick *random* bugs — it picks the worst one for each word. Robustness comes from training on *worst-case* perturbations (as in notebook 1), not from noise.
* **Sanitisation alone only closes the door it was built for.** PhantomText removes the homoglyph edits, and the adaptive attacker simply switches to visible typos (swaps, hyphens), so its success rate barely drops. Adding spell-correction closes that door too.
* **Fixing the representation beats fixing the classifier.** Sanitisation + spell-correction (undo confusables, map typos back into the vocabulary) removes most of the attack surface at zero accuracy cost; character n-grams help for the same reason.

No single defense is complete, and each costs something (compute, an engineering dependency on a vocabulary). Note that sanitisation is a **non-differentiable preprocessing step** — exactly the kind of defense that *looked* unbreakable for images until adaptive attacks arrived; here the attacker's search is already black-box, so the number above is honest.

## Take-aways

* Text attacks trade the ε-ball for **readability constraints**; the attacker's budget is "edits a human would not notice".
* **Word importance is cheap to estimate by querying** → black-box attacks are nearly as strong as white-box ones.
* **Universal triggers** are input-agnostic and the ancestor of prompt injection.
* Adversarial text **transfers** across models that share a tokenisation; changing the *representation* (char n-grams) helps more than changing the classifier.
* Detect and sanitise your inputs (confusables, zero-width, bidi — e.g. with PhantomText) — it is the text equivalent of input validation, and it is only as complete as its character tables.

### References
* Li et al., *TextBugger: Generating Adversarial Text Against Real-world Applications*, NDSS 2019.
* Ebrahimi et al., *HotFlip: White-Box Adversarial Examples for Text Classification*, ACL 2018.
* Jin et al., *Is BERT Really Robust? (TextFooler)*, AAAI 2020.
* Wallace et al., *Universal Adversarial Triggers for Attacking and Analyzing NLP*, EMNLP 2019.
* Boucher et al., *Bad Characters: Imperceptible NLP Attacks*, IEEE S&P 2022 (homoglyphs, invisible characters).
* Castagnaro, Salviati, Conti, Pajola, Pizzi, *The Hidden Threat in Plain Text: Attacking RAG Data Loaders*, arXiv:2507.05093, 2025 — and the [PhantomText](https://github.com/lucapajola/PhantomText) toolkit used for detection and sanitisation here.